# Part 4 — LLM-as-Judge by Hand

*Evals from First Principles, Part 4.*

Part 3 measured how much two humans disagree on the same gold labels. Here we
swap one of the graders for a MODEL: an LLM judge reads a rubric, reads a
candidate answer, and emits a verdict. The question is the same one we always
ask of a grader: how often does it agree with a trustworthy human, and where
exactly does it fail? We build a rubric prompt, run a deterministic mock judge
over 12 labeled items, parse its free text into PASS / FAIL / ABSTAIN, and
score it against human gold with the same 2x2 confusion matrix from Part 1.
Pure Python, offline, deterministic — no network, no API key, no real model.

## Step 1 — The rubric prompt

Before any judging happens we need to decide what we are asking the judge to
do. `RUBRIC` is the exact instruction we would hand a real LLM: reply on one
line as `VERDICT: PASS` or `VERDICT: FAIL`, fail any refusal, and pass only if
the required key fact is present. The mock judge below obeys exactly these
rules, deterministically, so every verdict it produces is traceable back to
this text.

In [ ]:
RUBRIC = """You are grading one answer. Reply on ONE line as: VERDICT: PASS
or VERDICT: FAIL.
  - FAIL if the answer refuses or declines to respond.
  - PASS only if the answer contains the required key fact.
  - Otherwise FAIL."""

line = "=" * 72
print(line)
print("PART 4 - LLM-AS-JUDGE BY HAND: scoring the judge against human gold.")
print(line)
print("The rubric we hand the judge:")
for r in RUBRIC.splitlines():
    print("  | " + r)


## Step 2 — The labeled set

Twelve items, each with a question, the KEY fact a good answer must contain,
the CANDIDATE answer under test, and the HUMAN gold grade (1 = a human said
PASS, 0 = FAIL). A human-written `note` names why each item is interesting —
display only, never used in any computation. Six of the twelve are clean human
PASS/FAIL calls; the rest are the edge cases (a terse correct answer, a
refusal, a keyword-gamed wrong answer, a verbose answer, two answers that are
correct via a synonym or a digit the judge will not recognize).

In [ ]:
ITEMS = [
    dict(qid="Q01", key="paris",    human=1,
         candidate="The capital of France is Paris.",
         note="clean pass"),
    dict(qid="Q02", key="tokyo",    human=1,
         candidate="Tokyo.",
         note="correct but terse -> TIE"),
    dict(qid="Q03", key="ignition", human=0,
         candidate="I cannot help with that request.",
         note="refusal"),
    dict(qid="Q04", key="jupiter",  human=1,
         candidate="The largest planet is Jupiter.",
         note="clean pass"),
    dict(qid="Q05", key="canberra", human=0,
         candidate="Sydney is the capital, though Canberra is the seat of parliament.",
         note="wrong, but the key word appears -> keyword-gamed"),
    dict(qid="Q06", key="washington", human=1,
         candidate="The answer is D.C.",
         note="correct via synonym the judge misses"),
    dict(qid="Q07", key="sphere",   human=1,
         candidate=("Well, if you really think about it in great detail, our "
                    "planet is not perfectly flat at all, and in fact people "
                    "have shown for many centuries that the earth is basically "
                    "a slightly squashed sphere."),
         note="verbose -> format drift"),
    dict(qid="Q08", key="pacific",  human=1,
         candidate="The Pacific Ocean is the largest.",
         note="clean pass"),
    dict(qid="Q09", key="five",     human=1,
         candidate="The result is 5.",
         note="correct as a digit the judge misses"),
    dict(qid="Q10", key="h2o",      human=1,
         candidate="Water is H2O.",
         note="clean pass"),
    dict(qid="Q11", key="brasilia", human=0,
         candidate="The capital of Brazil is Rio de Janeiro.",
         note="wrong answer, correctly failed"),
    dict(qid="Q12", key="four",     human=0,
         candidate="The answer is five.",
         note="wrong answer, correctly failed"),
]


## Step 3 — Detecting a refusal

`is_refusal` is a small keyword check the judge uses to fail any candidate
that declined to answer, independent of whether the key fact is present. It
matters as a rule because a refusal should never accidentally pass just for
mentioning the right word.

In [ ]:
REFUSAL_CUES = ("cannot", "can't", "unable", "i won't", "i will not")


def is_refusal(text):
    """True if the candidate declined to answer."""
    low = text.lower()
    return any(cue in low for cue in REFUSAL_CUES)


## Step 4 — The mock judge

`mock_judge` is a deterministic stand-in for an LLM judge: given the same
inputs it always returns the same raw text, so results are reproducible. It
returns the RAW verdict a model would emit, including the messy cases that
make real judges hard to trust: a TIE when there is too little content to
grade (fewer than 3 words), and format drift (no `VERDICT:` tag at all) when
the answer is long enough that the model rambles instead of following the
rubric's format.

In [ ]:
def mock_judge(candidate, key):
    """A deterministic stand-in for an LLM judge.

    It returns the RAW verdict STRING a model would emit, including the messy
    cases: a tie when there is too little to grade, and format drift (no
    VERDICT tag) when the answer is long enough to make the model ramble.
    """
    words = candidate.split()
    if is_refusal(candidate):
        return "VERDICT: FAIL  (the answer refuses to respond)"
    if len(words) < 3:
        return "VERDICT: TIE  (too little content to grade)"
    if len(words) > 25:
        # A long answer makes the model ramble and forget the format.
        return "After weighing both sides, this answer looks essentially fine to me."
    if key in candidate.lower():
        return "VERDICT: PASS"
    return "VERDICT: FAIL"


## Step 5 — Parsing the verdict, and two small helpers

`parse_verdict` turns the judge's free text into `1` (PASS), `0` (FAIL), or
`None` (ABSTAIN). ABSTAIN covers everything we cannot trust: a TIE token, or
any reply with no `VERDICT:` tag at all. `grade` and `shorten` are display
helpers: the first turns a 1/0 back into the word PASS/FAIL, the second
truncates a long raw verdict so the table below stays readable.

In [ ]:
def parse_verdict(raw):
    """Turn the judge's free text into 1 (PASS), 0 (FAIL), or None (ABSTAIN).

    ABSTAIN covers everything we cannot trust: a TIE token, or any reply with
    no 'VERDICT:' tag at all (format drift).
    """
    marker = "VERDICT:"
    if marker not in raw:
        return None
    token = raw.split(marker, 1)[1].strip().split()[0].strip(".").upper()
    if token == "PASS":
        return 1
    if token == "FAIL":
        return 0
    return None


def grade(word):
    return "PASS" if word == 1 else "FAIL"


def shorten(raw, width=42):
    return raw if len(raw) <= width else raw[:width].rstrip() + "..."


## Step 6 — Run the judge over every item

For each of the 12 items we call the judge, parse its verdict, and track two
things: whether the candidate was a refusal (display only), and whether the
parsed verdict is usable. Usable verdicts (PASS or FAIL) are appended to
`gold_scored` / `judge_scored` so they can later be scored against human gold;
ABSTAIN verdicts are dropped from that pair but still printed and still
counted as either a tie or format drift.

In [ ]:
print("\n" + "-" * 72)
print("Judge verdicts (raw model text -> parsed label):")
print("-" * 72)
gold_scored, judge_scored = [], []
refusals = ties = drift = 0
for it in ITEMS:
    raw = mock_judge(it["candidate"], it["key"])
    parsed = parse_verdict(raw)
    if is_refusal(it["candidate"]):
        refusals += 1
    if parsed is None:
        if "VERDICT:" not in raw:
            drift += 1
        else:
            ties += 1
        label = "ABSTAIN"
    else:
        label = grade(parsed)
        gold_scored.append(it["human"])
        judge_scored.append(parsed)
    print(f"  {it['qid']}  gold={grade(it['human'])}  judge->{label:<8}"
          f"  raw: {shorten(raw)}")


## Step 7 — Counting the failure modes

Three things shrink the evaluable set below all 12 items: a refusal (kept in
the matrix, since the judge correctly failed it), a tie (dropped — the judge
refused to commit), and format drift (dropped — no parseable tag at all).
`scored` is how many items actually make it into the confusion matrix.

In [ ]:
scored = len(gold_scored)
abstained = len(ITEMS) - scored
print("\n" + "-" * 72)
print("FAILURE MODES the parser had to survive:")
print("-" * 72)
print(f"  refusals (candidate declined) : {refusals}"
      "   -> judge said FAIL, kept in the matrix")
print(f"  ties     (judge won't commit) : {ties}"
      "   -> ABSTAIN, dropped from the matrix")
print(f"  format drift (no VERDICT tag) : {drift}"
      "   -> ABSTAIN, dropped from the matrix")
print(f"  scorable by the judge         : {scored} of {len(ITEMS)}"
      f"   ({abstained} abstained)")


## Step 8 — The 2x2 confusion matrix, from Part 1

`confusion` is the exact same function from Part 1: given the gold labels and
the judge's predictions on the scorable set, it counts the four cells of the
2x2 (positive class = PASS). Everything downstream — accuracy, precision,
recall, F1 — comes from just these four numbers.

In [ ]:
def confusion(gold, pred):
    """The four cells of the 2x2, positive class = PASS. From Part 1."""
    tp = fp = fn = tn = 0
    for g, p in zip(gold, pred):
        if g == 1 and p == 1:
            tp += 1
        elif g == 0 and p == 1:
            fp += 1
        elif g == 1 and p == 0:
            fn += 1
        else:
            tn += 1
    return tp, fp, fn, tn


## Step 9 — Judge vs human: the metrics

With `tp, fp, fn, tn` in hand, accuracy, precision, recall, and F1 follow
directly, same formulas as Part 1. The one false positive (Q05) is
keyword-gaming: a wrong answer that happened to contain the key word. The two
false negatives (Q06, Q09) are correct answers the judge failed for using a
synonym or a digit instead of the exact key. A judge is just another noisy
grader — measure it against humans before you trust its score.

In [ ]:
tp, fp, fn, tn = confusion(gold_scored, judge_scored)
n = tp + fp + fn + tn
acc = (tp + tn) / n
prec = tp / (tp + fp) if (tp + fp) else 0.0
rec = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
print("\n" + "-" * 72)
print("JUDGE vs HUMAN on the scorable set (positive = PASS):")
print("-" * 72)
print(f"  n = {n}  (human PASS: {tp + fn}, human FAIL: {fp + tn})")
print(f"  confusion:  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"  accuracy  = (TP+TN)/n  = ({tp}+{tn})/{n} = {acc:.2f}")
print(f"  precision = TP/(TP+FP) = {tp}/({tp}+{fp}) = {prec:.2f}")
print(f"  recall    = TP/(TP+FN) = {tp}/({tp}+{fn}) = {rec:.2f}")
print(f"  F1        = 2PR/(P+R)  = {f1:.2f}")
print()
print("  The one FP (Q05) is keyword-gaming: a wrong answer that happened to")
print("  contain the key word. The two FN (Q06, Q09) are correct answers the")
print("  judge failed for using a synonym or a digit. A judge is just another")
print("  noisy grader - measure it against humans before you trust its score.")
print(line)


## Recap

- An LLM judge is a grader, not an oracle. Score it against human gold with
  the same 2x2 confusion matrix and the same four metrics from Part 1
  (accuracy `0.70`, precision `0.80`, recall `0.67`, F1 `0.73`) before trusting
  its verdicts on anything you have not spot-checked.
- Real judges do not always return a clean label. Here 2 of 12 items abstained
  (a tie and a case of format drift), shrinking the evaluable set to 10 —
  build your parser to recognize and drop these rather than silently miscount
  them as a PASS or FAIL.
- Failure modes are informative, not just noise: the one false positive was
  keyword-gaming (the wrong answer happened to contain the key word), and both
  false negatives were correct answers the judge missed for using a synonym or
  a digit instead of the literal key. Knowing *why* a judge disagrees with a
  human is what tells you whether to fix the rubric, fix the parser, or accept
  the judge's noise floor.

Next: Part 5 asks the harder question — how much do we trust the judge itself,
and how do we judge the judge against a panel of humans.